# 🎯 AFA-Attack on NIPS 2017 Adversarial Learning Development Set
## Evaluation on 5 ImageNet Architectures

> **AFA-Attack** (Adaptive Frequency-Attention Attack) applied to the  
> [NIPS 2017 Adversarial Learning Dev Set](https://www.kaggle.com/datasets/google-brain/nips-2017-adversarial-learning-development-set)  
> — 1000 ImageNet images (299×299), L∞ budget ε = 16/255.

| Architecture   | Backbone           | Input Size | Source         |
|----------------|--------------------|------------|----------------|
| **inc_v3**     | Inception-v3       | 299×299    | timm ImageNet  |
| **inc_v4**     | Inception-v4       | 299×299    | timm ImageNet  |
| **inc_res_v2** | Inception-ResNet-v2| 299×299    | timm ImageNet  |
| **res_50**     | ResNet-50          | 224×224    | timm ImageNet  |
| **res_101**    | ResNet-101         | 224×224    | timm ImageNet  |

White-box model: **ResNet-50** | Transfer targets: all others


In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [3]:
!pip install kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [kaggle]2m3/4 [kaggle]dk]


In [4]:
import os

os.environ['KAGGLE_API_TOKEN'] = 'KGAT_eafe1bfc3280ec76feaeb276b28f45b4'

In [7]:
import json
import os

# create local kaggle folder (NOT /root)
os.makedirs('./.kaggle', exist_ok=True)

kaggle_config = {
    "username": "parthdhanker",
    "key": "KGAT_eafe1bfc3280ec76feaeb276b28f45b4"
}

# save locally
with open('./.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_config, f)

os.chmod('./.kaggle/kaggle.json', 0o600)

# tell kaggle where config is
os.environ['KAGGLE_CONFIG_DIR'] = os.path.abspath('./.kaggle')

print("✅ Kaggle configured (Lightning-safe)")

✅ Kaggle configured (Lightning-safe)


In [8]:
!kaggle datasets download -d google-brain/nips-2017-adversarial-learning-development-set -p ./nips2017 --unzip

Dataset URL: https://www.kaggle.com/datasets/google-brain/nips-2017-adversarial-learning-development-set
License(s): unknown
100%|█████████████████████████████████████████| 146M/146M [00:01<00:00, 121MB/s]



In [2]:
import os
import glob

DATASET_DIR = './nips2017'

imgs = glob.glob(f'{DATASET_DIR}/images/*')
print("Images:", len(imgs))

print("images.csv exists:", os.path.exists(f'{DATASET_DIR}/images.csv'))
print("categories.csv exists:", os.path.exists(f'{DATASET_DIR}/categories.csv'))

Images: 1000
images.csv exists: True
categories.csv exists: True


In [3]:
# ─────────────────────────────────────────
# SECTION 2 — Imports & Configuration
# ─────────────────────────────────────────
import os, math, time, copy, functools, warnings, glob
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Hyperparameters ───────────────────────────────────────────────────────────
CFG = {
    # AFA attack parameters
    "epsilon":      16 / 255,      # L∞ budget (16/255 is standard for ImageNet)
    "T":            10,             # Outer iterations
    "N":            10,             # Inner spectrum samples
    "rho":          0.5,            # Uniform mask range
    "sigma":        16 / 255,       # Gaussian noise std
    "image_size":   299,            # Input resolution fed to dataset loader
    # AFA-specific
    "tau":          1.0,
    "tau_min":      0.1,
    "lambda_coup":  0.15,
    "delta_conv":   0.05,
    # Experiment
    "num_images":   200,
    "batch_size":   8,             # small: 299×299 images are GPU-heavy
    "seed":         42,
    "dataset_dir":  "/content/nips2017",
}
CFG["alpha"] = CFG["epsilon"] / CFG["T"]

torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])

print("\n📋 CFG:")
for k, v in CFG.items():
    print(f"   {k:20s} = {v}")


🖥️  Device: cuda
   GPU : NVIDIA A100-SXM4-40GB
   VRAM: 42.4 GB

📋 CFG:
   epsilon              = 0.06274509803921569
   T                    = 10
   N                    = 10
   rho                  = 0.5
   sigma                = 0.06274509803921569
   image_size           = 299
   tau                  = 1.0
   tau_min              = 0.1
   lambda_coup          = 0.15
   delta_conv           = 0.05
   num_images           = 200
   batch_size           = 8
   seed                 = 42
   dataset_dir          = /content/nips2017
   alpha                = 0.006274509803921568


In [4]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import pandas as pd

# ─────────────────────────────────────────
# NIPS 2017 Dataset (FINAL CLEAN VERSION)
# ─────────────────────────────────────────

class NIPS2017Dataset(Dataset):
    def __init__(self, dataset_dir, transform=None, max_images=None):
        self.img_dir = os.path.join(dataset_dir, 'images')
        self.transform = transform

        # Load CSV
        df = pd.read_csv(os.path.join(dataset_dir, 'images.csv'))
        df.columns = [c.strip() for c in df.columns]

        # Standard column names (NIPS dataset)
        self.image_ids = df['ImageId'].astype(str).tolist()
        self.labels = (df['TrueLabel'].values.astype(int) - 1).tolist()

        if max_images:
            self.image_ids = self.image_ids[:max_images]
            self.labels = self.labels[:max_images]

        # Build full paths
        self.paths = [
            os.path.join(self.img_dir, img_id + '.png')
            for img_id in self.image_ids
        ]

        print(f"✅ Loaded {len(self.paths)} images")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img, self.labels[idx]


# ─────────────────────────────────────────
# CATEGORY NAMES
# ─────────────────────────────────────────
def load_category_names(dataset_dir):
    df = pd.read_csv(os.path.join(dataset_dir, 'categories.csv'))
    df.columns = [c.strip() for c in df.columns]
    return df['CategoryName'].tolist()


# ─────────────────────────────────────────
# TRANSFORM
# ─────────────────────────────────────────
base_transform = T.Compose([
    T.Resize((CFG['image_size'], CFG['image_size'])),
    T.ToTensor(),
])

# ─────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────
dataset = NIPS2017Dataset(
    './nips2017',   # ✅ correct path
    transform=base_transform,
    max_images=CFG['num_images']
)

LOADER = DataLoader(
    dataset,
    batch_size=CFG['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

CAT_NAMES = load_category_names('./nips2017')

# ─────────────────────────────────────────
# SANITY CHECK
# ─────────────────────────────────────────
print(f"Batches : {len(LOADER)}")
print(f"Classes : {len(CAT_NAMES)}")

x_sample, y_sample = next(iter(LOADER))
print(f"Batch shape : {x_sample.shape}")
print(f"Label range : {y_sample.min().item()} – {y_sample.max().item()}")
print(f"Pixel range : {x_sample.min():.3f} – {x_sample.max():.3f}")

✅ Loaded 200 images
Batches : 25
Classes : 1000
Batch shape : torch.Size([8, 3, 299, 299])
Label range : 243 – 990
Pixel range : 0.000 – 1.000


In [5]:
# ─────────────────────────────────────────
# SECTION 4 — NormalizedModel
#
# Wraps any backbone so attack code always feeds
# raw [0,1] tensors at image_size resolution.
# The wrapper handles:
#   1. Resize to backbone's native input_size
#   2. ImageNet normalisation (µ, σ per channel)
# ─────────────────────────────────────────

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class NormalizedModel(nn.Module):
    """
    Accepts raw [0,1] tensors of any spatial size.
    Resizes to `input_size` then applies ImageNet normalisation.
    """
    def __init__(self, backbone, mean=IMAGENET_MEAN, std=IMAGENET_STD,
                 input_size=224):
        super().__init__()
        self.backbone   = backbone
        self.input_size = input_size
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor(std ).view(1, 3, 1, 1))

    def forward(self, x):
        # Resize if needed
        if x.shape[-1] != self.input_size or x.shape[-2] != self.input_size:
            x = F.interpolate(x, size=(self.input_size, self.input_size),
                              mode='bilinear', align_corners=False)
        # Normalise
        x = (x - self.mean.to(x.device)) / self.std.to(x.device)
        return self.backbone(x)

print("✅ NormalizedModel defined (resizes + ImageNet normalisation)")


✅ NormalizedModel defined (resizes + ImageNet normalisation)


In [6]:
# GPU Memory Wipe — run before loading models
import gc, torch

if 'ALL_MODELS' in globals():
    for m in ALL_MODELS.values():
        if m is not None: del m
gc.collect()
torch.cuda.empty_cache()
print(f"✅ VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")


✅ VRAM free: 41.3 GB


In [7]:
# ─────────────────────────────────────────
# SECTION 5 — Load 5 ImageNet Pretrained Models
#
# All models loaded directly from timm with ImageNet weights.
# No fine-tuning needed — NIPS 2017 images are from ImageNet.
# ─────────────────────────────────────────
import gc, timm, torch, torch.nn as nn

ARCH_NAMES = ['inc_v3', 'inc_v4', 'inc_res_v2', 'res_50', 'res_101']
WHITE_BOX  = 'res_50'

# timm model name and native input size per arch
TIMM_CFG = {
    'inc_v3':     ('inception_v3',          299),
    'inc_v4':     ('inception_v4',          299),
    'inc_res_v2': ('inception_resnet_v2',   299),
    'res_50':     ('resnet50',              224),
    'res_101':    ('resnet101',             224),
}

def _free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def _vram_free():
    return torch.cuda.mem_get_info()[0] / 1e9

ALL_MODELS = {}
print("🔧 Loading ImageNet pretrained models via timm\n")
print(f"  {'Arch':14s}  {'timm name':28s}  {'Input':8s}  Status")
print("  " + "─" * 65)

for arch in ARCH_NAMES:
    timm_name, in_sz = TIMM_CFG[arch]
    try:
        _free_gpu()
        backbone = timm.create_model(timm_name, pretrained=True)
        backbone.eval()

        # Freeze all parameters — no fine-tuning
        for p in backbone.parameters():
            p.requires_grad_(False)

        wrapped = NormalizedModel(backbone, input_size=in_sz)
        wrapped.eval()
        # Keep on CPU; move to GPU per-batch during attack
        wrapped = wrapped.cpu()
        del backbone
        _free_gpu()

        ALL_MODELS[arch] = wrapped
        tag = " ◀ WHITE-BOX" if arch == WHITE_BOX else ""
        print(f"  {arch:14s}  {timm_name:28s}  {in_sz}×{in_sz}  ✅{tag}")

    except Exception as e:
        ALL_MODELS[arch] = None
        print(f"  {arch:14s}  {timm_name:28s}  {in_sz}×{in_sz}  ❌ {e}")

# ── Sanity check ──────────────────────────────────────────────────────────────
print("\n📊 Sanity check (CPU forward, raw 299×299 input):")
dummy = torch.randn(1, 3, CFG['image_size'], CFG['image_size'])
for arch, m in ALL_MODELS.items():
    if m is None:
        print(f"   {arch:14s}  ── load failed ──")
        continue
    with torch.no_grad():
        out = m(dummy)
    print(f"   {arch:14s}  input {list(dummy.shape)} → output {list(out.shape)}")

print(f"\n✅ Models ready. VRAM free: {_vram_free():.1f} GB")


🔧 Loading ImageNet pretrained models via timm

  Arch            timm name                     Input     Status
  ─────────────────────────────────────────────────────────────────


  inc_v3          inception_v3                  299×299  ✅
  inc_v4          inception_v4                  299×299  ✅
  inc_res_v2      inception_resnet_v2           299×299  ✅
  res_50          resnet50                      224×224  ✅ ◀ WHITE-BOX
  res_101         resnet101                     224×224  ✅

📊 Sanity check (CPU forward, raw 299×299 input):


[W414 10:42:08.560458881 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.589937788 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.590521855 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.593376148 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.602287041 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.602877738 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.603547118 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.606175496 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.615786008 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.616627853 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:4

   inc_v3          input [1, 3, 299, 299] → output [1, 1000]


[W414 10:42:08.799387350 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.814107553 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.852761939 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.854117350 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.856748446 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.867979786 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.869347557 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.870584303 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.873189790 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.884405720 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:4

...

[W414 10:42:08.227365729 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.255723443 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.256509407 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.260058061 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.267852710 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.268498509 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.268950362 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.270176388 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.272851457 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:08.274974850 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:4

   inc_res_v2      input [1, 3, 299, 299] → output [1, 1000]
   res_50          input [1, 3, 299, 299] → output [1, 1000]


[W414 10:42:09.632657552 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.638650847 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.643457500 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.647760226 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.649332823 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.653384231 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.654951027 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.659013097 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.661658724 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.667373773 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:4

   res_101         input [1, 3, 299, 299] → output [1, 1000]

✅ Models ready. VRAM free: 41.3 GB


[W414 10:42:09.837444225 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.838959820 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.843154063 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.844693518 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.848864681 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.850394617 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.854565659 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.856086014 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.860254076 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:42:09.861762611 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W414 10:4

In [8]:
# ─────────────────────────────────────────
# SECTION 6a — Differentiable 2D DCT / IDCT
# Identical to original AFA notebook.
# Gradients flow back through T_AFA → D_I → D → x.
# ─────────────────────────────────────────

@functools.lru_cache(maxsize=16)
def _dct_matrix(n: int, device_str: str) -> torch.Tensor:
    k = torch.arange(n, dtype=torch.float32)
    i = torch.arange(n, dtype=torch.float32)
    A = torch.cos(math.pi / n * (i.unsqueeze(0) + 0.5) * k.unsqueeze(1))
    A[0] *= math.sqrt(1.0 / n)
    A[1:] *= math.sqrt(2.0 / n)
    return A.to(device_str)

def dct1d(x: torch.Tensor) -> torch.Tensor:
    n   = x.shape[-1]
    A   = _dct_matrix(n, str(x.device))
    return x @ A.T

def idct1d(X: torch.Tensor) -> torch.Tensor:
    n   = X.shape[-1]
    A   = _dct_matrix(n, str(X.device))
    return X @ A

def dct2d(x: torch.Tensor) -> torch.Tensor:
    return dct1d(dct1d(x).transpose(-2, -1)).transpose(-2, -1)

def idct2d(X: torch.Tensor) -> torch.Tensor:
    return idct1d(idct1d(X).transpose(-2, -1)).transpose(-2, -1)

print("✅ Differentiable 2D DCT/IDCT ready")


✅ Differentiable 2D DCT/IDCT ready


In [9]:
# ─────────────────────────────────────────
# SECTION 6b — YCbCr Utilities
# Identical to original AFA notebook.
# ─────────────────────────────────────────

_RGB2YCBCR = torch.tensor([
    [ 0.29900,  0.58700,  0.11400],
    [-0.16874, -0.33126,  0.50000],
    [ 0.50000, -0.41869, -0.08131],
], dtype=torch.float32)
_YCBCR2RGB = torch.inverse(_RGB2YCBCR)


def rgb_to_ycbcr(x: torch.Tensor) -> torch.Tensor:
    M = _RGB2YCBCR.to(x.device)
    return torch.einsum('bchw,cd->bdhw', x, M.T)


def ycbcr_to_rgb(x: torch.Tensor) -> torch.Tensor:
    M = _YCBCR2RGB.to(x.device)
    return torch.einsum('bchw,cd->bdhw', x, M.T)


def cross_channel_coupling(freq: torch.Tensor, lambda_val: float = 0.15
                            ) -> torch.Tensor:
    """Orthogonal rotation mixing Y and Cb frequency components."""
    theta    = lambda_val * math.pi / 2
    cos_t, sin_t = math.cos(theta), math.sin(theta)
    Y, Cb, Cr = freq[:, 0:1], freq[:, 1:2], freq[:, 2:3]
    Y_new  = cos_t * Y - sin_t * Cb
    Cb_new = sin_t * Y + cos_t * Cb
    return torch.cat([Y_new, Cb_new, Cr], dim=1)


print("✅ YCbCr / cross-channel coupling ready")


✅ YCbCr / cross-channel coupling ready


In [10]:
# ─────────────────────────────────────────
# SECTION 6c — GradCAM Implementation
# Identical to original AFA notebook.
# ─────────────────────────────────────────

class GradCAM:
    """
    GradCAM for any timm/torchvision model.
    Hooks the last Conv2d layer.
    """
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model    = model
        self.fwd_acts = None
        self.bwd_grads = None
        self._hooks   = [
            target_layer.register_forward_hook(self._save_act),
            target_layer.register_full_backward_hook(self._save_grad),
        ]

    def _save_act(self, _, __, out):
        self.fwd_acts = out.detach()

    def _save_grad(self, _, __, grad_out):
        self.bwd_grads = grad_out[0].detach()

    def remove_hooks(self):
        for h in self._hooks: h.remove()

    def compute(self, x, y):
        """Returns (B, 1, H, W) saliency map in [0, 1]."""
        x_in = x.detach().requires_grad_(True)
        logits = self.model(x_in)
        loss   = F.cross_entropy(logits, y)
        self.model.zero_grad()
        loss.backward()

        weights = self.bwd_grads.mean(dim=[-2, -1], keepdim=True)
        cam     = (weights * self.fwd_acts).sum(dim=1, keepdim=True)
        cam     = F.relu(cam)
        cam     = F.interpolate(cam, size=x.shape[-2:],
                                mode='bilinear', align_corners=False)
        cam_min = cam.flatten(1).min(1)[0].view(-1,1,1,1)
        cam_max = cam.flatten(1).max(1)[0].view(-1,1,1,1)
        return (cam - cam_min) / (cam_max - cam_min + 1e-8)


def get_last_conv(model: nn.Module) -> nn.Module:
    """Returns the last Conv2d in a model."""
    last = None
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    return last


def build_gradcam(arch, wrapped_model):
    """Build GradCAM on the last Conv2d of the backbone inside NormalizedModel."""
    backbone = wrapped_model.backbone
    last_conv = get_last_conv(backbone)
    if last_conv is None:
        print(f"   ⚠️  {arch}: no Conv2d found — GradCAM disabled")
        return None
    return GradCAM(wrapped_model, last_conv)


# Build GradCAM for white-box model
GRADCAM = None
if ALL_MODELS.get(WHITE_BOX) is not None:
    wb = ALL_MODELS[WHITE_BOX]
    GRADCAM = build_gradcam(WHITE_BOX, wb)
    status = "✅" if GRADCAM else "⚠️ disabled"
    print(f"GradCAM on '{WHITE_BOX}': {status}")


GradCAM on 'res_50': ✅


In [11]:
# ─────────────────────────────────────────
# SECTION 7a — AFA-Attack Core Components
# S_φ, V, and Ω — the three adaptive steering masks.
# Identical to original AFA notebook.
# ─────────────────────────────────────────

def compute_spectrum_saliency(model, x, y):
    """S_φ = DCT(∂J/∂x) — frequency-domain gradient."""
    x_in = x.detach().clone().requires_grad_(True)
    loss = F.cross_entropy(model(x_in), y)
    loss.backward()
    return dct2d(x_in.grad.detach())


def compute_vulnerability_mask(S_phi, tau=1.0):
    """V = softmax(|S_φ| / τ) — high where model is sensitive."""
    B, C, H, W = S_phi.shape
    abs_flat   = (S_phi.abs() / (tau + 1e-8)).reshape(B, -1)
    return F.softmax(abs_flat, dim=-1).reshape(B, C, H, W)


def compute_anti_attention_mask(x, y, gradcam):
    """Ω = 1 − normalize(DCT(GradCAM)) — high where model is NOT attending."""
    A      = gradcam.compute(x, y)          # (B, 1, H, W) in [0,1]
    A_3ch  = A.expand(-1, 3, -1, -1)
    G      = dct2d(A_3ch).abs()
    B      = G.shape[0]
    G_flat = G.reshape(B, -1)
    G_min  = G_flat.min(1)[0].reshape(B,1,1,1)
    G_max  = G_flat.max(1)[0].reshape(B,1,1,1)
    return 1.0 - (G - G_min) / (G_max - G_min + 1e-8)


print("✅ AFA core components: S_φ | V | Ω")


✅ AFA core components: S_φ | V | Ω


In [12]:
# ─────────────────────────────────────────
# SECTION 7b — T_AFA Transformation
# Identical to original AFA notebook.
# ─────────────────────────────────────────

def T_AFA(x, V, Omega, sigma=16/255, rho=0.5, lambda_val=0.15):
    """
    T_AFA(x) = D_I^{YCbCr→RGB}(
                  C_λ · [(D^{RGB→YCbCr}(x) + V⊙ξ') ⊙ (1 + Ω⊙m)]
               )
    """
    x_ycbcr   = rgb_to_ycbcr(x)
    freq      = dct2d(x_ycbcr)
    xi_guided = V * (torch.randn_like(freq) * sigma)
    freq      = freq + xi_guided
    m_guided  = Omega * (torch.rand_like(freq) * 2 * rho - rho)
    freq      = freq * (1.0 + m_guided)
    freq      = cross_channel_coupling(freq, lambda_val=lambda_val)
    return ycbcr_to_rgb(idct2d(freq))


print("✅ T_AFA transformation ready (Steps A–F)")


✅ T_AFA transformation ready (Steps A–F)


In [13]:
# ─────────────────────────────────────────
# SECTION 7c-NEW — Transferability Helper Functions (MI / DI / TI)
#
# MI (Momentum Integration) — stabilises gradient direction; escapes
#    model-specific saddle points that block transfer.
# DI (Input Diversity)      — random resize+pad breaks the white-box
#    model's feature reliance before each forward pass.
# TI (Translation Invariance) — Gaussian gradient smoothing removes
#    spatially sharp, model-specific noise from the update direction.
# ─────────────────────────────────────────

def create_gaussian_kernel(kernel_size=15, nsig=3.0, channels=3):
    """(channels, 1, ks, ks) Gaussian kernel for depthwise conv."""
    coords = torch.arange(kernel_size, dtype=torch.float32) - kernel_size // 2
    g1d = torch.exp(-coords ** 2 / (2 * nsig ** 2))
    g1d = g1d / g1d.sum()
    g2d = g1d.outer(g1d)
    return g2d.view(1, 1, kernel_size, kernel_size).repeat(channels, 1, 1, 1)


def smooth_grad_ti(grad, kernel):
    """TI: depthwise Gaussian blur of gradient tensor."""
    ks = kernel.shape[-1]
    return F.conv2d(grad, kernel.to(grad.device), padding=ks // 2, groups=grad.shape[1])


def input_diversity(x, p=0.5, scale_low=0.90):
    """DI: random resize+zero-pad back to original (H, W)."""
    if torch.rand(1).item() >= p:
        return x
    B, C, H, W = x.shape
    rnd_h = torch.randint(int(scale_low * H), H + 1, (1,)).item()
    rnd_w = torch.randint(int(scale_low * W), W + 1, (1,)).item()
    x_rs  = F.interpolate(x, size=(rnd_h, rnd_w), mode='bilinear', align_corners=False)
    top   = torch.randint(0, H - rnd_h + 1, (1,)).item()
    left  = torch.randint(0, W - rnd_w + 1, (1,)).item()
    return F.pad(x_rs, [left, W-rnd_w-left, top, H-rnd_h-top], value=0)


# Pre-build TI kernel (reused every batch)
TI_KERNEL = create_gaussian_kernel(kernel_size=15, nsig=3.0, channels=3)

print('✅ Transferability helpers ready: MI | DI | TI')
print(f'   TI kernel shape: {list(TI_KERNEL.shape)}')


✅ Transferability helpers ready: MI | DI | TI
   TI kernel shape: [3, 1, 15, 15]


In [14]:
# ─────────────────────────────────────────
# SECTION 7d — AFA-Attack v2  (MI + DI + TI)
#
# Identical to the original Algorithm 2 EXCEPT for three additions
# that break model-specificity and dramatically improve black-box transfer:
#
#  MI : momentum buffer `g` replaces raw avg_grad in the sign update.
#       Gradient is L1-normalised before accumulation so momentum is
#       scale-consistent across iterations.
#
#  DI : after T_AFA, each of the N inner samples is passed through
#       random resize+pad before the model forward.  This prevents
#       the gradient from overfitting the white-box receptive field.
#
#  TI : the N-averaged gradient is Gaussian-blurred before the MI
#       update.  This smooths out high-frequency, model-specific
#       gradient artefacts that do not survive cross-architecture transfer.
# ─────────────────────────────────────────

def afa_attack_v2(model, x_orig, y, gradcam,
                   epsilon=16/255, T=10, N=10,
                   alpha=16/255/10,
                   rho=0.5, sigma=16/255,
                   tau=1.0, tau_min=0.1,
                   lambda_val=0.15, delta_conv=0.05,
                   # ── Transferability knobs ──
                   mu=1.0,        # MI momentum decay (1.0 = full carry)
                   use_di=True,   # DI: enable input diversity
                   di_prob=0.7,   # DI: apply probability per inner sample
                   ti_kernel=None # TI: Gaussian kernel tensor (or None)
                   ):

    x_adv       = x_orig.clone()
    current_tau = tau
    g           = torch.zeros_like(x_orig)   # MI: momentum buffer

    for t in range(T):
        # ── Compute adaptive frequency masks (unchanged from v1) ──────────
        S_phi = compute_spectrum_saliency(model, x_adv, y)
        V     = compute_vulnerability_mask(S_phi, tau=current_tau)
        Omega = (compute_anti_attention_mask(x_adv, y, gradcam)
                 if gradcam is not None else torch.ones_like(x_adv))

        # ── N inner samples with DI ───────────────────────────────────────
        grad_accum = torch.zeros_like(x_adv)
        for _ in range(N):
            x_t  = x_adv.clone().requires_grad_(True)

            # AFA frequency-domain transform (unchanged)
            x_tr = torch.clamp(
                T_AFA(x_t, V=V.detach(), Omega=Omega.detach(),
                      sigma=sigma, rho=rho, lambda_val=lambda_val),
                0.0, 1.0)

            # DI: random resize+pad before model (NEW)
            if use_di:
                x_tr = input_diversity(x_tr, p=di_prob)

            loss = F.cross_entropy(model(x_tr), y)
            loss.backward()
            grad_accum = grad_accum + x_t.grad.detach()

        avg_grad = grad_accum / N

        # TI: smooth gradient with Gaussian kernel (NEW)
        if ti_kernel is not None:
            avg_grad = smooth_grad_ti(avg_grad, ti_kernel)

        # MI: L1-normalise then accumulate momentum (NEW)
        avg_grad = avg_grad / (avg_grad.abs().mean() + 1e-8)
        g        = mu * g + avg_grad

        # ── L∞-projected sign step (uses momentum g instead of raw grad) ─
        with torch.no_grad():
            x_proposed = torch.clamp(
                torch.max(
                    torch.min(x_adv + alpha * g.sign(), x_orig + epsilon),
                    x_orig - epsilon),
                0.0, 1.0)

        # ── Adaptive tau (unchanged from v1) ─────────────────────────────
        S_phi_new  = compute_spectrum_saliency(model, x_proposed, y)
        frob_shift = (S_phi_new - S_phi).norm(p='fro').item()

        if frob_shift < delta_conv:
            current_tau = max(current_tau * 0.5, tau_min)
            V = compute_vulnerability_mask(S_phi, tau=current_tau)

        x_adv = x_proposed.detach()

    return x_adv


print('✅ AFA-Attack v2 (MI + DI + TI) fully implemented')


✅ AFA-Attack v2 (MI + DI + TI) fully implemented


In [15]:
# ─────────────────────────────────────────
# SECTION 8a — Evaluation Utilities (updated for v2)
# ─────────────────────────────────────────

import numpy as np
from tqdm import tqdm


def get_predictions(model, x):
    dev = next(model.parameters()).device
    with torch.no_grad():
        return model(x.to(dev)).argmax(dim=1).cpu()


def run_experiment_v2(loader, models_dict, white_box_name, gradcam, cfg,
                       mu=1.0, use_di=True, di_prob=0.7, ti_kernel=None):
    wb_model  = models_dict[white_box_name].cuda()
    wb_device = next(wb_model.parameters()).device

    # Rebuild GradCAM hooks on the CUDA model
    if gradcam is not None:
        gradcam.remove_hooks()
        gradcam = build_gradcam(white_box_name, wb_model)

    # Move TI kernel to GPU once
    ti_k = ti_kernel.to(wb_device) if ti_kernel is not None else None

    results = {'afa_v2': {name: {'hits': 0, 'total': 0} for name in models_dict}}
    pbar = tqdm(loader, desc='  AFA-v2 (MI+DI+TI)', unit='batch')

    for x_clean, y_true in pbar:
        x_clean = x_clean.to(wb_device)
        y_true  = y_true.to(wb_device)

        correct_mask = (get_predictions(wb_model, x_clean).to(wb_device) == y_true)
        if correct_mask.sum() == 0:
            continue
        x_clean = x_clean[correct_mask]
        y_true  = y_true[correct_mask]

        x_adv = afa_attack_v2(
            model=wb_model, x_orig=x_clean, y=y_true,
            gradcam=gradcam,
            epsilon=cfg['epsilon'], T=cfg['T'],  N=cfg['N'],
            alpha=cfg['alpha'],    rho=cfg['rho'], sigma=cfg['sigma'],
            tau=cfg['tau'],        tau_min=cfg['tau_min'],
            lambda_val=cfg['lambda_coup'], delta_conv=cfg['delta_conv'],
            mu=mu, use_di=use_di, di_prob=di_prob, ti_kernel=ti_k,
        )

        n_batch = y_true.shape[0]
        for name, model in models_dict.items():
            if name == white_box_name:
                preds = get_predictions(wb_model, x_adv)
            else:
                model_dev = model.cuda()
                preds = get_predictions(model_dev, x_adv)
                model_dev.cpu()

            n_success = (preds != y_true.cpu()).sum().item()
            results['afa_v2'][name]['hits']  += n_success
            results['afa_v2'][name]['total'] += n_batch

        torch.cuda.empty_cache()
        wb_asr = (results['afa_v2'][white_box_name]['hits'] /
                  max(results['afa_v2'][white_box_name]['total'], 1)) * 100
        pbar.set_postfix(wb_asr=f'{wb_asr:.1f}%')

    final = {}
    for name in models_dict:
        d = results['afa_v2'][name]
        final[name] = (d['hits'] / d['total'] * 100.0) if d['total'] > 0 else 0.0
    results['afa_v2'] = final

    wb_model.cpu()
    torch.cuda.empty_cache()
    return results


In [17]:
# ─────────────────────────────────────────
# SECTION 8b — Run AFA v2 on NIPS 2017 and compare with v1
# ─────────────────────────────────────────

import time, numpy as np

active_models = {k: v for k, v in ALL_MODELS.items() if v is not None}

print(f"\n{'='*60}")
print(f"  DATASET : NIPS 2017 (ImageNet dev-set)")
print(f"  White-box: {WHITE_BOX}  |  eps = {CFG['epsilon']*255:.0f}/255")
print(f"  T={CFG['T']}  N={CFG['N']}  Images={CFG['num_images']}  BS={CFG['batch_size']}")
print(f"  Boosts: MI(mu=1.0) + DI(p=0.7) + TI(ks=15)")
print(f"{'='*60}")

t0 = time.time()
ALL_RESULTS = run_experiment_v2(
    loader=LOADER, models_dict=active_models,
    white_box_name=WHITE_BOX, gradcam=GRADCAM, cfg=CFG,
    mu=1.0, use_di=True, di_prob=0.7, ti_kernel=TI_KERNEL,
)
elapsed = time.time() - t0

black_models = [k for k in active_models if k != WHITE_BOX]

print(f"\n  {'Model':16s}  {'v1 (original)':>14s}  {'v2 (MI+DI+TI)':>14s}  {'Delta':>7s}  Role")
print(f"  {'-'*68}")
for name in ARCH_NAMES:
    if name not in ALL_RESULTS_V2['afa_v2']:
        continue
    v1  = ALL_RESULTS.get('afa', {}).get(name, float('nan'))
    v2  = ALL_RESULTS_V2['afa_v2'][name]
    tag = '  White-box' if name == WHITE_BOX else '  Black-box'
    dlt = f'+{v2-v1:.1f}%' if v1 == v1 else 'n/a'
    print(f"  {name:16s}  {v1:>13.1f}%  {v2:>13.1f}%  {dlt:>7s}{tag}")

print(f"  {'-'*68}")
if black_models:
    bb_v1 = np.mean([ALL_RESULTS.get('afa',{}).get(n, 0) for n in black_models
                     if n in ALL_RESULTS.get('afa', {})])
    bb_v2 = np.mean([ALL_RESULTS_V2['afa_v2'][n]
                     for n in black_models if n in ALL_RESULTS_V2['afa_v2']])
    print(f"  {'Black-box AVG':16s}  {bb_v1:>13.1f}%  {bb_v2:>13.1f}%  +{bb_v2-bb_v1:.1f}%")

print(f"\n  Total elapsed: {elapsed:.0f}s")



  DATASET : NIPS 2017 (ImageNet dev-set)
  White-box: res_50  |  eps = 16/255
  T=10  N=10  Images=200  BS=8
  Boosts: MI(mu=1.0) + DI(p=0.7) + TI(ks=15)


  AFA-v2 (MI+DI+TI): 100%|██████████| 25/25 [01:03<00:00,  2.53s/batch, wb_asr=98.4%]


  Model              v1 (original)   v2 (MI+DI+TI)    Delta  Role
  --------------------------------------------------------------------
  inc_v3                      nan%           53.4%      n/a  Black-box
  inc_v4                      nan%           51.9%      n/a  Black-box
  inc_res_v2                  nan%           40.7%      n/a  Black-box
  res_50                      nan%           98.9%      n/a  White-box
  res_101                     nan%           60.8%      n/a  Black-box
  --------------------------------------------------------------------
  Black-box AVG               nan%           51.7%  +nan%

  Total elapsed: 63s


In [ ]:
# ─────────────────────────────────────────
# SECTION 8c — Side-by-Side Bar Chart: AFA v1 vs v2
# ─────────────────────────────────────────

import matplotlib.pyplot as plt, numpy as np

model_names = [n for n in ARCH_NAMES if n in ALL_RESULTS_V2['afa_v2']]
v1_vals = [ALL_RESULTS.get('afa', {}).get(n, 0) for n in model_names]
v2_vals = [ALL_RESULTS_V2['afa_v2'][n] for n in model_names]

x = np.arange(len(model_names))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x - w/2, v1_vals, w, label='AFA v1 (original)',
             color='steelblue', alpha=0.85, edgecolor='navy')
b2 = ax.bar(x + w/2, v2_vals, w, label='AFA v2 (MI+DI+TI)',
             color='crimson',  alpha=0.85, edgecolor='darkred')

for bar, val in zip(b1, v1_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=8, color='navy')
for bar, val in zip(b2, v2_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=8, color='darkred')

ax.set_xticks(x)
ax.set_xticklabels([f"{n}\n({'WB' if n==WHITE_BOX else 'BB'})" for n in model_names], fontsize=10)
ax.set_ylim(0, 115)
ax.set_ylabel('Attack Success Rate (%)', fontsize=11)
ax.set_title(
    f'AFA v1 vs AFA v2 (MI+DI+TI)  -  NIPS 2017  (eps={CFG["epsilon"]*255:.0f}/255, WB={WHITE_BOX})\n'
    f'{CFG["num_images"]} images  |  T={CFG["T"]}  N={CFG["N"]}',
    fontsize=12, fontweight='bold')
ax.axhline(np.mean(v1_vals), color='steelblue', ls='--', alpha=0.5,
           label=f'v1 mean: {np.mean(v1_vals):.1f}%')
ax.axhline(np.mean(v2_vals), color='crimson',  ls='--', alpha=0.5,
           label=f'v2 mean: {np.mean(v2_vals):.1f}%')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
import os; os.makedirs('results', exist_ok=True)
save_path = 'results/NIPS2017_AFA_v1_vs_v2.png'
plt.savefig(save_path, bbox_inches='tight', dpi=300)
plt.show()
print(f'Saved to {save_path}')


In [ ]:
# ─────────────────────────────────────────
# SECTION 8d — Visual: Clean vs AFA-v2 Adversarial
# (GradCAM hooks are rebuilt on the CUDA model — fixes the AttributeError)
# ─────────────────────────────────────────

def visualize_attack_comparison(clean_imgs, adv_imgs, true_labels,
                                  clean_preds, adv_preds, class_names,
                                  num_images=5,
                                  save_path='results/AFA_v2_vis.png', dpi=150):
    n = min(num_images, len(clean_imgs))
    fig, axes = plt.subplots(3, n, figsize=(3*n, 9))
    fig.suptitle('AFA v2 (MI+DI+TI) — Clean vs Adversarial',
                 fontsize=13, fontweight='bold')

    def to_np(t):
        return t.permute(1,2,0).clamp(0,1).numpy()

    for i in range(n):
        axes[0,i].imshow(to_np(clean_imgs[i]))
        axes[0,i].set_title(f'Clean\n{class_names[true_labels[i]][:14]}', fontsize=7)
        axes[0,i].axis('off')

        axes[1,i].imshow(to_np(adv_imgs[i]))
        pname = class_names[adv_preds[i]] if adv_preds[i] < len(class_names) else str(adv_preds[i])
        col = 'red' if adv_preds[i] != true_labels[i] else 'green'
        axes[1,i].set_title(f'Adversarial\n{pname[:14]}', fontsize=7, color=col)
        axes[1,i].axis('off')

        axes[2,i].imshow(to_np((adv_imgs[i]-clean_imgs[i]).abs() * 5))
        axes[2,i].set_title('Perturbation (x5)', fontsize=7)
        axes[2,i].axis('off')

    for row, lbl in enumerate(['Clean', 'Adversarial', 'Perturbation (x5)']):
        axes[row,0].set_ylabel(lbl, fontsize=9, rotation=90, labelpad=40)

    plt.tight_layout()
    import os; os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
    plt.savefig(save_path, bbox_inches='tight', dpi=dpi)
    plt.show()
    print(f'Saved to {save_path}')


# Move model to CUDA and rebuild GradCAM (avoids 'NoneType has no attr mean')
wb_model = ALL_MODELS[WHITE_BOX].cuda()
wb_model.eval()
wb_dev = next(wb_model.parameters()).device

if GRADCAM is not None:
    GRADCAM.remove_hooks()
gradcam_cuda = build_gradcam(WHITE_BOX, wb_model)

images, labels = next(iter(LOADER))
images, labels = images.to(wb_dev), labels.to(wb_dev)

with torch.no_grad():
    clean_preds = wb_model(images).argmax(dim=1)

adv_images = afa_attack_v2(
    model=wb_model, x_orig=images, y=labels,
    gradcam=gradcam_cuda,
    epsilon=CFG['epsilon'], T=CFG['T'],  N=CFG['N'],
    alpha=CFG['alpha'],    rho=CFG['rho'], sigma=CFG['sigma'],
    tau=CFG['tau'],        tau_min=CFG['tau_min'],
    lambda_val=CFG['lambda_coup'], delta_conv=CFG['delta_conv'],
    mu=1.0, use_di=True, di_prob=0.7, ti_kernel=TI_KERNEL.to(wb_dev),
).detach()

with torch.no_grad():
    adv_preds = wb_model(adv_images).argmax(dim=1)

wb_model.cpu()
torch.cuda.empty_cache()

visualize_attack_comparison(
    clean_imgs=images.cpu(), adv_imgs=adv_images.cpu(),
    true_labels=labels.cpu().tolist(),
    clean_preds=clean_preds.cpu().tolist(),
    adv_preds=adv_preds.cpu().tolist(),
    class_names=CAT_NAMES,
    num_images=5,
    save_path='results/NIPS2017_AFA_v2_vis.png',
)
